# Homework 2 — train a Transformer from scratch

This notebook uses only the supplied `SlavaYus/tinysms_qa_12m` training split. The T5-style encoder–decoder starts with random weights; the included tokenizer is fitted on the same training split. Run this in Colab with a **T4 GPU**. Checkpoints are saved in the Colab runtime; download `latest/` before the runtime ends. The 20,000-update run is an experiment target, not a claim of achieved quality.

In [1]:
!git clone --depth=1 --branch=2026 https://github.com/OsVox/hse-nlp.git /content/hse-nlp
%cd /content/hse-nlp
!pip -q install 'transformers>=4.53,<5' 'pyarrow>=18' 'requests>=2.31'

Cloning into '/content/hse-nlp'...
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 32 (delta 0), reused 25 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (32/32), 26.81 MiB | 108.07 MiB/s, done.
/content/hse-nlp
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 188.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 172.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have hu

In [2]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')
assert torch.cuda.is_available(), 'Select T4 GPU in Runtime → Change runtime type'

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [3]:
CHECKPOINT_DIR = '/content/hw2_checkpoints'
print('Checkpoint directory:', CHECKPOINT_DIR)

Checkpoint directory: /content/hw2_checkpoints


In [4]:
!python -m homeworks.hw2.train --steps 20000 --batch-size 16 --accumulation 4 --cache-dir /content/hw2_cache --output-dir "$CHECKPOINT_DIR" --save-every 500 --eval-every 500

2026-10-03 08:44:15.410008: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-03 08:44:15.441585: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
device=cuda
parameters=9,967,360
step=1 train_loss=9.6342 elapsed_s=2.6
step=50 train_loss=7.2806 elapsed_s=6.6
step=100 train_loss=5.2033 elapsed_s=10.7
step=150 train_loss=4.5124 elapsed_s=14.7
step=200 train_loss=4.3369 elapsed_s=18.8
step=250 train_loss=4.2096 elapsed_s=23.0
step=300 train_loss=3.9725 elapsed_s=27.1
step=350 train_loss=3.7433 elap

In [5]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
checkpoint = Path('/content/hw2_checkpoints/latest')
assert (checkpoint / 'model.safetensors').exists()
archive_path = Path('/content/hw2_baseline_model.zip')
with ZipFile(archive_path, 'w') as archive:
    for path in checkpoint.iterdir():
        if path.is_file() and path.name != 'training_state.pt':
            archive.write(path, arcname=path.name)
print('Baseline checkpoint:', [(p.name, p.stat().st_size) for p in checkpoint.iterdir() if p.is_file() and p.name != 'training_state.pt'])
files.download(str(archive_path))

Baseline checkpoint: [('config.json', 684), ('special_tokens_map.json', 416), ('generation_config.json', 142), ('step.json', 15), ('tokenizer.json', 545550), ('model.safetensors', 39880744), ('tokenizer_config.json', 780)]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
!git pull --ff-only && python -m homeworks.hw2.evaluate --model-dir /content/hw2_checkpoints/latest --cache-dir /content/hw2_cache --rows 256 --offset 256

remote: Enumerating objects: 58, done.
remote: Counting objects: 100% (58/58), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 48 (delta 39), reused 34 (delta 25), pack-reused 0 (from 0)
Unpacking objects: 100% (48/48), 6.75 KiB | 1.69 MiB/s, done.
From https://github.com/OsVox/hse-nlp
   00352d9..f104221  2026       -> origin/2026
Updating 00352d9..f104221
Fast-forward
 homeworks/hw2/README.md                 | 20 ++++++++++++++------
 homeworks/hw2/data.py                   | 15 ++++++++++++---
 homeworks/hw2/evaluate.py               | 20 ++++++++++++++++----
 homeworks/hw2/hw2_inference_colab.ipynb | 10 ++++++++++
 homeworks/hw2/hw2_train_colab.ipynb     | 30 +++++++++++++++++++++++++++---
 homeworks/hw2/inference.py              |  8 ++++----
 homeworks/hw2/train.py                  | 27 +++++++++++++++++++++++++--
 7 files changed, 108 insertions(+), 22 deletions(-)
2026-10-03 09:15:20.090479: I tensorflow/core/util/port.cc:153] oneDNN custom operations are o

In [7]:
!python -m homeworks.hw2.train --steps 20000 --batch-size 16 --accumulation 4 --batches-per-shard 1000 --cache-dir /content/hw2_cache --output-dir /content/hw2_rotated --save-every 500 --eval-every 500

2026-10-03 09:16:24.210724: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-03 09:16:24.242829: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
device=cuda
parameters=9,967,360
step=1 train_loss=9.6342 elapsed_s=0.7
step=50 train_loss=7.2806 elapsed_s=4.8
step=100 train_loss=5.2033 elapsed_s=8.9
step=150 train_loss=4.5124 elapsed_s=12.9
step=200 train_loss=4.3370 elapsed_s=17.0
step=250 train_loss=4.2096 elapsed_s=20.9
step=300 train_loss=3.9297 elapsed_s=25.5
step=350 train_loss=3.7128 elaps

In [8]:
!python -m homeworks.hw2.evaluate --model-dir /content/hw2_rotated/best --cache-dir /content/hw2_cache --rows 256 --offset 256

2026-10-03 09:46:00.999882: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-03 09:46:01.032003: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
{
  "validation_rows": 256,
  "validation_offset": 256,
  "syntax_valid_fraction": 0.99609375,
  "solve_defined_fraction": 0.99609375,
  "exact_match_fraction": 0.0078125,
  "note": "These are diagnostics, not pass@1; no test answers are available."
}


In [9]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
root = Path('/content/hw2_rotated/best')
archive = Path('/content/hw2_rotated_best_model.zip')
with ZipFile(archive, 'w') as bundle:
    for name in ['config.json', 'generation_config.json', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'validation.json']:
        path = root / name
        if path.exists():
            bundle.write(path, name)
print(archive, archive.stat().st_size)
files.download(str(archive))

/content/hw2_rotated_best_model.zip 40429165


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import ast, torch
from pathlib import Path
from transformers import PreTrainedTokenizerFast, T5ForConditionalGeneration
from homeworks.hw2.data import validation_examples
root = Path('/content/hw2_rotated/best')
tok = PreTrainedTokenizerFast.from_pretrained(root)
model = T5ForConditionalGeneration.from_pretrained(root).to('cuda').eval()
examples = validation_examples(Path('/content/hw2_cache'))[256:512]
predictions = []
with torch.inference_mode():
    for start in range(0, len(examples), 32):
        batch = tok([row['question'] for row in examples[start:start+32]], padding=True, truncation=True, max_length=128, return_tensors='pt', return_token_type_ids=False).to('cuda')
        generated = model.generate(**batch, max_new_tokens=192, num_beams=4)
        predictions.extend(tok.batch_decode(generated, skip_special_tokens=True))
valid = 0
for text in predictions:
    try:
        ast.parse(text)
        valid += bool(text.strip())
    except SyntaxError:
        pass
print('beam4_valid', valid, 'exact', sum(a.strip() == b['code'].strip() for a,b in zip(predictions, examples)))

beam4_valid 255 exact 1


The last cell prints training loss and held-out token loss. Download the `latest/` model files before Colab disconnects, then copy the model files to `homeworks/hw2/model/` in the fork for the inference notebook. `training_state.pt` is only needed to resume training. Colab runtime storage is temporary.